# M3 Program 3: Dynamic Programming

CSC3310

Names: Madison Betz, Chukwuma Chukwuma-Ugwu, Sam Schoenfeld

## Approach
To solve the MAXIMALSUBSTRING problem, we identify the longest string that is a contiguous substring of both input strings a and b. Our strategy follows a three-stage evolution: starting with a recursive backtracking algorithm to test potential character matches, moving to a memoized version that stores the results in an array to eliminate redundant calculations, and finally implementing an eager dynamic programming solution. The core logic focuses on substrings ending at a specific pair of characters (i,j). If the characters a[i] and b[j] match, the length of the common substring is 1+ the value of the diagonal neighbor (i - 1, j - 1); otherwise, the length is 0. By iteratively filling a 2D table using this rule, the algorithm efficiently finds the maximum length and returns the corresponding substring.

### Illustration 1
This diagram visualizes how the value of each cell in the dynamic programming table is derived. A match at the current indices (i,j) steps up the value from the previous diagonal state, representing the continuation of a common substring.

![logic](logic.png)

### Illustration 2
This table demonstrates the "eager" filling process. The algorithm tracks the highest value in the table and its location to get the final result.

![table](table.png)

## Pseudocode

### Backtracking (Recursive)
Method LCS_Backtrack(a, b, i, j)
1. If i < 0 or j < 0, return 0
2. If a[i] == b[j]: return 1 + LCS_Backtrack(a, b, i - 1, j - 1)
3. Else: return 0

### Memoized (Lazy Dynamic Programming)
Initialize memo table of size n x m with -1 <br>
Algorithm LCS_Memo(a, b, i, j)
1. If i < 0 or j < 0 return 0
2.    If memo[i][j] != -1: return memo[i][j]
3. If a[i] == b[j]: memo[i][j] = 1 + LCS_Memo(a, b, i - 1, j - 1)
4. Else: memo[i][j] = 0
5. Return memo[i][j]

### Eager Dynamic Programming
Algorithm MAX_SUBSTRING(a, b)
1. Let n = |a|, m = |b|
2. Create a 2D array Table of size (n + 1) x (m + 1) initialized to 0
3. maxLength = 0, endIndex = 0
4. For i from 1 to n: For j from 1 to m:
5. If a[i - 1] == b[j - 1]: Table[i][j] = Table[i - 1][j - 1] + 1
6. If Table[i][j] > maxLength: maxLength = Table[i][j], endIndex = i
7. Else: Table[i][j] = 0
8. Return the slice of a from endIndex - maxLength to endIndex


In [ ]:
def lcs_backtrack(a, b, i, j):
    if i < 0 or j < 0:
        return 0
    if a[i] == b[j]:
        return 1 + lcs_backtrack(a, b, i - 1, j - 1)
    return 0

def get_maximal_backtrack(a, b):
    max_len = 0
    end_idx = 0
    for i in range(len(a)):
        for j in range(len(b)):
            current_len = lcs_backtrack(a, b, i, j)
            if current_len > max_len:
                max_len = current_len
                end_idx = i + 1
    return a[end_idx - max_len : end_idx]

In [ ]:
def get_maximal_memoized(a, b):
    n, m = len(a), len(b)
    memo = [[-1 for _ in range(m)] for _ in range(n)]

    def lcs_memo(i, j):
        if i < 0 or j < 0:
            return 0
        if memo[i][j] != -1:
            return memo[i][j]

        if a[i] == b[j]:
            memo[i][j] = 1 + lcs_memo(i - 1, j - 1)
        else:
            memo[i][j] = 0
        return memo[i][j]

    max_len = 0
    end_idx = 0
    for i in range(n):
        for j in range(m):
            current_len = lcs_memo(i, j)
            if current_len > max_len:
                max_len = current_len
                end_idx = i + 1
    return a[end_idx - max_len : end_idx]

In [ ]:
def get_maximal_eager_dp(a, b):
    n, m = len(a), len(b)
    table = [[0 for _ in range(m + 1)] for _ in range(n + 1)]
    max_len = 0
    end_idx = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i-1] == b[j-1]:
                table[i][j] = table[i-1][j-1] + 1
                if table[i][j] > max_len:
                    max_len = table[i][j]
                    end_idx = i
            else:
                table[i][j] = 0

    return a[end_idx - max_len : end_idx]

## Justification of Correctness

The backtracking algorithm is correct because `lcs_backtrack(a, b, i, j)` returns exactly the length of the longest common substring ending at index `i` in `a` and index `j` in `b`. This can be proven by induction on the sum `i + j`. The base case holds: when either index is negative, no substring is possible, so 0 is returned. For the inductive step, if `a[i] == b[j]`, the two characters form a common endpoint and the problem reduces to extending whatever shared suffix ended at `(i-1, j-1)` — which the recursive call computes correctly by the inductive hypothesis. If `a[i] != b[j]`, no common substring can end at both `i` and `j`, so 0 is returned. The outer loops in `get_maximal_backtrack` exhaustively try every possible ending pair `(i, j)`, and the maximum over all pairs gives the globally longest common substring, which is reconstructed via a slice.

The memoized algorithm is correct for the same reason: it computes the identical recurrence, and memoization only caches results to avoid redundant recomputation. Because each call to `lcs_memo(i, j)` depends only on `lcs_memo(i-1, j-1)` — strictly "earlier" in the diagonal — there are no circular dependencies, and every subproblem is evaluated exactly once. The value retrieved from or stored in the memo table always matches what plain recursion would return, so correctness is fully inherited from the backtracking argument.

The eager dynamic programming algorithm is correct by induction on the order in which the table is filled. The invariant is that `table[i][j]` holds the length of the longest common substring of `a` and `b` that ends exactly at `a[i-1]` and `b[j-1]`. The boundary (row 0 and column 0) is initialized to 0, satisfying the base case. When the table is filled row-by-row, `table[i-1][j-1]` is always computed before `table[i][j]`, so the diagonal dependency is always satisfied. If `a[i-1] == b[j-1]`, the value extends the diagonal; otherwise it resets to 0, exactly matching the recurrence. Maintaining `maxLength` and `endIndex` at every step ensures the globally maximal value and its location are captured, and the final slice correctly reconstructs the substring.


## Test Cases

The table below shows each test case, the expected output, and the actual results from all three algorithm implementations. For the "pinaeapple" / "orange" case, the expected result is any single character from the set {`"n"`, `"a"`, `"e"`} — the algorithm returns the first maximal match it encounters.

| String a | String b | Expected | Backtrack | Memoized | Eager DP |
|---|---|---|---|---|---|
| `"orange"` | `"banana"` | `"an"` | `"an"` | `"an"` | `"an"` |
| `"banana"` | `"apple"` | `"a"` | `"a"` | `"a"` | `"a"` |
| `"apple"` | `"pineapple"` | `"apple"` | `"apple"` | `"apple"` | `"apple"` |
| `"pinaeapple"` | `"orange"` | `"n"`, `"a"`, or `"e"` | `"n"` | `"n"` | `"n"` |
| `"abc"` | `"xyz"` | `""` | `""` | `""` | `""` |
| `"aaa"` | `"aa"` | `"aa"` | `"aa"` | `"aa"` | `"aa"` |
| `"a"` | `"a"` | `"a"` | `"a"` | `"a"` | `"a"` |

The code cell below confirms these results programmatically.


In [ ]:
test_cases = [
    ("orange", "banana", "an"),
    ("banana", "apple", "a"),
    ("apple", "pineapple", "apple"),
    ("pinaeapple", "orange", None), 
    ("abc", "xyz", ""),
    ("aaa", "aa", "aa"),
    ("a", "a", "a"),
]

print(f"{'String a':<14} {'String b':<12} {'Expected':<14} {'Backtrack':<12} {'Memoized':<12} {'Eager DP'}")
print("-" * 76)
for a, b, expected in test_cases:
    bt    = get_maximal_backtrack(a, b)
    memo  = get_maximal_memoized(a, b)
    eager = get_maximal_eager_dp(a, b)
    exp_str = "len=1 match" if expected is None else repr(expected)
    ok = all([
        (len(bt) >= 1 if expected is None else bt == expected),
        (len(memo) >= 1 if expected is None else memo == expected),
        (len(eager) >= 1 if expected is None else eager == expected),
    ])
    status = "PASS" if ok else "FAIL"
    print(f"{repr(a):<14} {repr(b):<12} {exp_str:<14} {repr(bt):<12} {repr(memo):<12} {repr(eager):<12} {status}")


## Asymptotic Runtime

**Backtracking** (`get_maximal_backtrack`)

The outer double loop over all pairs $(i, j)$ runs in $O(nm)$ iterations. At each pair, `lcs_backtrack` recurses diagonally along the $(i-1, j-1)$ chain. In the worst case (all characters match), this chain has depth $\min(i, j) \leq \min(n, m)$, with $O(1)$ work per call. Therefore:

$$T(n, m) = O\!\left(nm \cdot \min(n, m)\right)$$

**Memoized** (`get_maximal_memoized`)

There are $n \cdot m$ distinct subproblems. Each subproblem `lcs_memo(i, j)` is computed at most once — subsequent accesses are an $O(1)$ table lookup — and each computation does $O(1)$ work (one character comparison plus one recursive call that immediately returns from the memo). The outer loops add another $O(nm)$ factor, which does not change the overall bound:

$$T(n, m) = O(nm)$$

**Eager Dynamic Programming** (`get_maximal_eager_dp`)

The algorithm fills an $(n+1) \times (m+1)$ table with two nested loops. Each of the $O(nm)$ cells requires exactly one character comparison and one table read — both $O(1)$. There is no recursion overhead or memoization dictionary involved:

$$T(n, m) = O(nm)$$

**Summary:** Eager DP and Memoized are both $O(nm)$, matching the $\Omega(nm)$ lower bound on this problem (every cell corresponds to a potentially necessary character comparison). Eager DP is faster in practice due to lower constant factors. Backtracking is strictly worse at $O(nm \cdot \min(n, m))$, effectively cubic when $n \approx m$.


## Benchmarking
TODO: A table and graph from benchmarking different lists with different sizes and values of m and n. Compare the runtimes
for each algorithm.
If the benchmarks do not support your theoretically-derived run time and/or do not provide evidence that the run time
of your algorithm grows more slowly than the brute-force approach, this may indicate a flaw in your implementation.